In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import re
from data_prep import clean_price_pre_meter, convert_bool_to_int

df = pd.read_csv("../data/kufar_ads.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10150 entries, 0 to 10149
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ad_id                10150 non-null  int64  
 1   subject              10150 non-null  object 
 2   price_usd            10150 non-null  int64  
 3   price_per_meter_usd  10150 non-null  float64
 4   rooms                10150 non-null  int64  
 5   area_total           10150 non-null  float64
 6   area_living          6102 non-null   float64
 7   area_kitchen         2682 non-null   float64
 8   bathroom_type        5648 non-null   object 
 9   balcony_type         6220 non-null   object 
 10  has_balcony          10150 non-null  bool   
 11  is_first_floor       10150 non-null  bool   
 12  is_last_floor        10150 non-null  bool   
 13  year_built           10007 non-null  float64
 14  condition            10150 non-null  object 
 15  address              10150 non-null 

In [3]:
df.head(5)

,ad_id,subject,price_usd,price_per_meter_usd,rooms,area_total,area_living,area_kitchen,bathroom_type,balcony_type,has_balcony,is_first_floor,is_last_floor,year_built,condition,address,ad_link,list_time,first_seen_at,last_seen_at
0,1082532620,Продажа стильной евродвушки в Минск-Мир. Лос-А...,123000,2922.00,2,42.1,38.5,NaN,Совмещенный,Лоджия,True,False,False,2021.0,Вторичное,"Леонида Левина ул, 13, Минск",https://re.kufar.by/vi/1082532620,2026-08-25 19:13:33+00:00,2026-08-29 16:57:54.053523+00:00,2026-08-29 16:57:54.053523+00:00
1,1082532665,"2-комнатная квартира с видом на парк и водоем,...",152000,2171.00,2,70.0,36.0,12.0,Раздельный,Есть,True,False,False,2003.0,Вторичное,"Мазурова ул, 18, Минск",https://re.kufar.by/vi/1082532665,2026-08-25 19:13:17+00:00,2026-08-29 16:57:54.053523+00:00,2026-08-29 16:57:54.053523+00:00
2,1082532601,"Адрес, который повышает статус. Старый аэропор...",60461,1772.93,1,34.1,NaN,NaN,Совмещенный,Лоджия,True,False,False,2027.0,Новое,"площадь Старый Аэропорт, Минск",https://re.kufar.by/vi/1082532601,2026-08-25 19:12:30+00:00,2026-08-29 16:57:54.053523+00:00,2026-08-29 16:57:54.053523+00:00
3,1082532163,Не просто квадратные метры. Новые возможности...,60461,1772.93,1,34.1,NaN,NaN,Совмещенный,Лоджия,True,False,False,2027.0,Новое,"площадь Старый Аэропорт, Минск",https://re.kufar.by/vi/1082532163,2026-08-25 19:06:49+00:00,2026-08-29 16:57:54.053523+00:00,2026-08-29 16:57:54.053523+00:00
4,1082531458,Дом Лира Напрямую от застройщика РАССРОЧКА. Зв...,51070,1837.08,1,27.8,22.0,NaN,Совмещенный,NaN,False,False,False,2027.0,Новое,"площадь Старый Аэропорт, 2, Минск",https://re.kufar.by/vi/1082531458,2026-08-25 18:58:51+00:00,2026-08-29 16:57:54.053523+00:00,2026-08-29 16:57:54.053523+00:00


In [4]:
# Разделение колонок по типам данных
col_num = df.select_dtypes(include=['number']).columns.tolist()
col_str = df.select_dtypes(include=['object', 'string']).columns.tolist()

# Проверка результатов
print(f"(col_num), Len { len(col_num)}: {col_num}")
print(f"(col_str), Len {len(col_str)}: {col_str}")


(col_num), Len 8: ['ad_id', 'price_usd', 'price_per_meter_usd', 'rooms', 'area_total', 'area_living', 'area_kitchen', 'year_built']
(col_str), Len 9: ['subject', 'bathroom_type', 'balcony_type', 'condition', 'address', 'ad_link', 'list_time', 'first_seen_at', 'last_seen_at']


In [5]:
df[col_num].agg(['min', 'max', 'mean', 'median']).round(2)

,ad_id,price_usd,price_per_meter_usd,rooms,area_total,area_living,area_kitchen,year_built
min,1.153161e+08,0.00,0.00,1.00,1.00,5.0,1.00,1950.00
max,1.082533e+09,2300000.00,11011.00,5.00,638.90,773.0,83.60,2029.00
mean,1.069511e+09,104854.02,2049.89,1.53,49.64,38.0,11.94,2017.28
median,1.078224e+09,78806.00,1884.90,1.00,44.00,31.2,9.00,2027.00


In [6]:
pd.DataFrame({col: pd.Series(df[col].nunique()) for col in col_str}).fillna('-')


,subject,bathroom_type,balcony_type,condition,address,ad_link,list_time,first_seen_at,last_seen_at
0,5551,4,4,2,2213,10150,7605,339,339


In [7]:
pd.DataFrame({col: pd.Series(df[col].unique()) for col in col_str}).fillna('-')


,subject,bathroom_type,balcony_type,condition,address,ad_link,list_time,first_seen_at,last_seen_at
0,Продажа стильной евродвушки в Минск-Мир. Лос-А...,Совмещенный,Лоджия,Вторичное,"Леонида Левина ул, 13, Минск",https://re.kufar.by/vi/1082532620,2026-08-25 19:13:33+00:00,2026-08-29 16:57:54.053523+00:00,2026-08-29 16:57:54.053523+00:00
1,"2-комнатная квартира с видом на парк и водоем,...",Раздельный,Есть,Новое,"Мазурова ул, 18, Минск",https://re.kufar.by/vi/1082532665,2026-08-25 19:13:17+00:00,2026-08-29 16:57:54.089488+00:00,2026-08-29 16:57:54.089488+00:00
2,"Адрес, который повышает статус. Старый аэропор...",-,-,-,"площадь Старый Аэропорт, Минск",https://re.kufar.by/vi/1082532601,2026-08-25 19:12:30+00:00,2026-08-29 16:57:54.097277+00:00,2026-08-29 16:57:54.097277+00:00
3,Не просто квадратные метры. Новые возможности...,Два,Два,-,"площадь Старый Аэропорт, 2, Минск",https://re.kufar.by/vi/1082532163,2026-08-25 19:06:49+00:00,2026-08-29 16:57:55.661071+00:00,2026-08-29 16:57:55.661071+00:00
4,Дом Лира Напрямую от застройщика РАССРОЧКА. Зв...,Три,Нет,-,"Жореса Алфёрова ул, 1, Минск",https://re.kufar.by/vi/1082531458,2026-08-25 18:58:51+00:00,2026-08-29 16:57:54.104387+00:00,2026-08-29 16:57:54.104387+00:00
...,...,...,...,...,...,...,...,...,...
10145,-,-,-,-,-,https://re.kufar.by/vi/1044383052,-,-,-
10146,-,-,-,-,-,https://re.kufar.by/vi/1044383053,-,-,-
10147,-,-,-,-,-,https://re.kufar.by/vi/1070636180,-,-,-
10148,-,-,-,-,-,https://re.kufar.by/vi/1065264643,-,-,-


In [8]:
corr_matrix = df[col_num].corr(method='pearson')
corr_matrix['price_usd'].sort_values(ascending=False)

price_usd              1.000000
area_total             0.785800
rooms                  0.666969
area_living            0.666655
price_per_meter_usd    0.641873
area_kitchen           0.215326
ad_id                 -0.118244
year_built            -0.248053
Name: price_usd, dtype: float64

In [9]:
comparison  = df[["price_per_meter_usd", "area_total", "price_usd"]].copy()
comparison["calc_price"] = comparison["area_total"] * comparison["price_per_meter_usd"]
comparison.head(20)

,price_per_meter_usd,area_total,price_usd,calc_price
0,2922.00,42.1,123000,123016.200
1,2171.00,70.0,152000,151970.000
2,1772.93,34.1,60461,60456.913
3,1772.93,34.1,60461,60456.913
4,1837.08,27.8,51070,51070.824
5,1774.09,34.1,60478,60496.469
6,1839.41,30.8,56636,56653.828
7,3130.61,32.4,101431,101431.764
8,1952.64,39.5,77129,77129.280
9,1354.19,48.5,65664,65678.215


In [10]:
(df["price_per_meter_usd"] < 100).sum()

np.int64(31)

In [11]:
df = clean_price_pre_meter(df)

In [12]:
(df["area_total"] < 5).sum()


np.int64(0)

In [13]:
df = convert_bool_to_int(df)

Сконвертировано колонок: ['has_balcony', 'is_first_floor', 'is_last_floor']


In [14]:
def parse_address(address):
    parts = [p.strip() for p in address.split(",")]
    
    street = parts[0]
    house_number = None
    
    # ищем часть, которая похожа на номер дома (цифры, возможно с буквой/дробью типа "18/2", "5А")
    for p in parts[1:]:
        if re.match(r'^\d+[а-яА-Я]?(/\d+)?$', p):
            house_number = p
            break
    
    return pd.Series({"street": street, "house_number": house_number})

df[["street", "house_number"]] = df["address"].apply(parse_address)



In [15]:
df.head(150)

,ad_id,subject,price_usd,price_per_meter_usd,rooms,area_total,area_living,area_kitchen,bathroom_type,balcony_type,...,is_last_floor,year_built,condition,address,ad_link,list_time,first_seen_at,last_seen_at,street,house_number
0,1082532620,Продажа стильной евродвушки в Минск-Мир. Лос-А...,123000,2922.00,2,42.1,38.5,NaN,Совмещенный,Лоджия,...,0,2021.0,Вторичное,"Леонида Левина ул, 13, Минск",https://re.kufar.by/vi/1082532620,2026-08-25 19:13:33+00:00,2026-08-29 16:57:54.053523+00:00,2026-08-29 16:57:54.053523+00:00,Леонида Левина ул,13
1,1082532665,"2-комнатная квартира с видом на парк и водоем,...",152000,2171.00,2,70.0,36.0,12.0,Раздельный,Есть,...,0,2003.0,Вторичное,"Мазурова ул, 18, Минск",https://re.kufar.by/vi/1082532665,2026-08-25 19:13:17+00:00,2026-08-29 16:57:54.053523+00:00,2026-08-29 16:57:54.053523+00:00,Мазурова ул,18
2,1082532601,"Адрес, который повышает статус. Старый аэропор...",60461,1772.93,1,34.1,NaN,NaN,Совмещенный,Лоджия,...,0,2027.0,Новое,"площадь Старый Аэропорт, Минск",https://re.kufar.by/vi/1082532601,2026-08-25 19:12:30+00:00,2026-08-29 16:57:54.053523+00:00,2026-08-29 16:57:54.053523+00:00,площадь Старый Аэропорт,None
3,1082532163,Не просто квадратные метры. Новые возможности...,60461,1772.93,1,34.1,NaN,NaN,Совмещенный,Лоджия,...,0,2027.0,Новое,"площадь Старый Аэропорт, Минск",https://re.kufar.by/vi/1082532163,2026-08-25 19:06:49+00:00,2026-08-29 16:57:54.053523+00:00,2026-08-29 16:57:54.053523+00:00,площадь Старый Аэропорт,None
4,1082531458,Дом Лира Напрямую от застройщика РАССРОЧКА. Зв...,51070,1837.08,1,27.8,22.0,NaN,Совмещенный,NaN,...,0,2027.0,Новое,"площадь Старый Аэропорт, 2, Минск",https://re.kufar.by/vi/1082531458,2026-08-25 18:58:51+00:00,2026-08-29 16:57:54.053523+00:00,2026-08-29 16:57:54.053523+00:00,площадь Старый Аэропорт,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,1077885312,"4-комнатная квартира 89м² по СНБ, комнаты расп...",141588,1620.00,4,87.4,59.9,8.7,Раздельный,Лоджия,...,0,1981.0,Вторичное,"Рокоссовского пр, 18к1, Минск",https://re.kufar.by/vi/1077885312,2026-08-25 14:14:20+00:00,2026-08-29 16:57:54.113089+00:00,2026-08-29 16:57:54.113089+00:00,Рокоссовского пр,None
146,1066281516,"Минск, Нововиленская ул., д. 61, 3-комн.",331991,4322.81,3,76.8,63.5,NaN,Раздельный,Лоджия,...,0,2024.0,Новое,"ЖК Левада, Минск",https://re.kufar.by/vi/1066281516,2026-08-25 14:13:59+00:00,2026-08-29 16:57:54.113089+00:00,2026-08-29 16:57:54.113089+00:00,ЖК Левада,None
147,1033707074,"Минск, Тимирязева ул., д. 126, 2-комн.",191146,3068.17,2,62.3,37.1,NaN,Раздельный,Лоджия,...,0,2023.0,Новое,"Тимирязева ул, 126, Минск",https://re.kufar.by/vi/1033707074,2026-08-25 14:12:57+00:00,2026-08-29 16:57:54.113089+00:00,2026-08-29 16:57:54.113089+00:00,Тимирязева ул,126
148,1081976804,Рассрочка. Взнос. Узнай график платежей прямо ...,49412,1534.98,1,32.2,NaN,NaN,NaN,NaN,...,0,2027.0,Новое,"Николы Теслы ул, 33, Минск",https://re.kufar.by/vi/1081976804,2026-08-25 14:10:59+00:00,2026-08-29 16:57:54.113089+00:00,2026-08-29 16:57:54.113089+00:00,Николы Теслы ул,33


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10119 entries, 0 to 10149
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ad_id                10119 non-null  int64  
 1   subject              10119 non-null  object 
 2   price_usd            10119 non-null  int64  
 3   price_per_meter_usd  10119 non-null  float64
 4   rooms                10119 non-null  int64  
 5   area_total           10119 non-null  float64
 6   area_living          6093 non-null   float64
 7   area_kitchen         2674 non-null   float64
 8   bathroom_type        5626 non-null   object 
 9   balcony_type         6199 non-null   object 
 10  has_balcony          10119 non-null  int64  
 11  is_first_floor       10119 non-null  int64  
 12  is_last_floor        10119 non-null  int64  
 13  year_built           9985 non-null   float64
 14  condition            10119 non-null  object 
 15  address              10119 non-null  obje

In [17]:
# Числовые (min/max)
num_cols = ['rooms', 'year_built', 'area_total', 'area_living', 'area_kitchen']
print("--- NUMERICAL ---")
print(df[num_cols].agg(['min', 'max']))

# Все категории (включая твои подготовленные улицы)
cat_cols = ['bathroom_type', 'balcony_type', 'condition', 'street']
print("\n--- CATEGORICAL ---")
for col in cat_cols:
    print(f"{col} =", sorted(df[col].dropna().unique().tolist()))

--- NUMERICAL ---
     rooms  year_built  area_total  area_living  area_kitchen
min      1      1950.0         5.1          5.0           1.0
max      5      2029.0       638.9        773.0          83.6

--- CATEGORICAL ---
bathroom_type = ['Два', 'Раздельный', 'Совмещенный', 'Три']
balcony_type = ['Два', 'Есть', 'Лоджия', 'Нет']
condition = ['Вторичное', 'Новое']
street = ['1-й Осенний переулок', '1-й Подольский переулок', '1-й Полиграфический переулок', '1-й Радиаторный переулок', '1-я Поселковая ул', '1-я Радиаторная ул', '2-й Брагинский переулок', '2-й Измайловский переулок', '2-й Прилукский переулок', '2-й Сморговский переулок', '2-й Холмогорский переулок', '2-й переулок Багратиона', '2-й переулок Кольцова', '2-й переулок Никитина', '2-й переулок Тимошенко', '2-й переулок Халтурина', '2-я  Щорса ул', '3 Сентября ул', '3-й Железнодорожный переулок', '3-й Парниковый переулок', '3-й Поселковый переулок', '3-й переулок Зубачёва', '3-й переулок Можайского', '4-й Загородный переулок', 